<a href="https://colab.research.google.com/github/majorrip/POC-for-asynchronous-non-blocking-checkpointing/blob/main/POC_for_asynchronous_non_blocking_checkpointing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import time
import queue
import threading
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Ensure CUDA availability for asynchronous stream execution
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    print(" WARNING: CUDA is not available. Running on CPU. "
          "Asynchronous streams will execute synchronously.")

In [ ]:
# 1. Heterogeneous Non-Uniform Network
# ==========================================
class HeterogeneousDeepNet(nn.Module):
    """
    A non-uniform model architecture designed with irregular memory structures
    (alternating 2D/1D layers, pooling, and residual projections) to simulate
    complex state topologies typical in advanced deep architectures.
    """
    def __init__(self):
        super().__init__()
        # Irregular structural layers
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # Heavy deep bottleneck with asymmetric channels
        self.conv2 = nn.Conv2d(64, 192, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(192, 512, kernel_size=3, padding=1)

        # Global pooling and dimensionality flattening
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4)) # 512 * 4 * 4 = 8192

        # Irregular transition to fully connected layers
        self.fc1 = nn.Linear(8192, 4096)
        self.fc2 = nn.Linear(4096, 1024)
        self.fc3 = nn.Linear(1024, 10)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
# 2. Asynchronous Composable State Provider
# ==========================================
class AsyncStateProvider:
    """
    Implements a non-blocking checkpoint state provider that orchestrates
    asynchronous D2H copies via dedicated CUDA streams and stages disk writes
    to a background worker thread.
    """
    def __init__(self, model, optimizer, simulated_io_delay=0.4):
        self.model = model
        self.optimizer = optimizer
        self.io_delay = simulated_io_delay

        # Dedicated CUDA stream for non-blocking Device-to-Host transfers
        self.checkpoint_stream = torch.cuda.Stream() if torch.cuda.is_available() else None

        # Thread safe queue to pass staged CPU states to the storage engine thread
        self.io_queue = queue.Queue()
        self.worker_thread = threading.Thread(target=self._io_worker, daemon=True)
        self.worker_thread.start()

    def _clone_to_pinned_cpu(self, tensor):
        """
        Allocates CPU memory and schedules an asynchronous copy.
        Safely falls back to standard pageable CPU memory if a CUDA allocator isn't present.
        """
        # Check if the tensor is actually on a GPU and CUDA is active
        is_cuda = tensor.is_cuda

        # Only request pinned memory if we are running on an active accelerator backend
        pinned_tensor = torch.empty(
            tensor.shape,
            dtype=tensor.dtype,
            layout=tensor.layout,
            pin_memory=is_cuda  # Dynamically toggles True/False
        )

        # If CUDA is active, this is a non-blocking DMA transfer. Otherwise, it's a standard copy.
        pinned_tensor.copy_(tensor, non_blocking=is_cuda)
        return pinned_tensor

    def trigger_async_checkpoint(self, epoch_idx):
        """
        Triggers a non-blocking checkpoint. Captures state tensors asynchronously
        without blocking the main execution thread.
        """
        # Context manager switches stream execution to our dedicated CP stream
        stream_context = torch.cuda.stream(self.checkpoint_stream) if self.checkpoint_stream else DummyContext()

        with stream_context:
            staged_model_state = {}
            staged_opt_state = {}

            # 1. Asynchronously stage model weights to pinned CPU memory
            for k, v in self.model.state_dict().items():
                staged_model_state[k] = self._clone_to_pinned_cpu(v)

            # 2. Asynchronously stage Adam optimizer state tensors (e.g., exp_avg, exp_avg_sq)
            for param, state_param in self.optimizer.state.items():
                param_id = id(param)
                staged_opt_state[param_id] = {}
                for state_key, state_val in state_param.items():
                    if isinstance(state_val, torch.Tensor):
                        staged_opt_state[param_id][state_key] = self._clone_to_pinned_cpu(state_val)
                    else:
                        staged_opt_state[param_id][state_key] = state_val

            # Ensure CPU synchronizes ONLY with the checkpoint stream, not the main compute stream
            if self.checkpoint_stream:
                self.checkpoint_stream.synchronize()

        # Hand over the pinned memory payload to the background disk writer thread
        self.io_queue.put((epoch_idx, staged_model_state, staged_opt_state))

    def _io_worker(self):
        """Background thread executing decoupled storage serialization (Disk I/O)."""
        while True:
            item = self.io_queue.get()
            if item is None:
                break
            epoch_idx, _, _ = item
            # Simulate heavy serialization and disk write delay
            time.sleep(self.io_delay)
            self.io_queue.task_done()

    def join(self):
        """Blocks until all pending background storage I/O operations complete."""
        self.io_queue.join()


class DummyContext:
    """Fallback context manager for CPU execution environments."""
    def __enter__(self): pass
    def __exit__(self, exc_type, exc_val, exc_tb): pass

In [ ]:
# 3. Training Loop Simulations
# ==========================================
def run_simulation(strategy="baseline", num_iterations=8, io_delay=0.4):
    print(f"\n🚀 Initializing Benchmarking Run: [{strategy.upper()} STRATEGY]")

    # Instantiate architecture, multi-state optimizer, and dummy data
    model = HeterogeneousDeepNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    # Mock data dimensions representing structured batched inputs
    inputs = torch.randn(64, 3, 224, 224, device=device)
    targets = torch.randint(0, 10, (64,), device=device)

    # Initialize state provider if using the asynchronous method
    async_provider = AsyncStateProvider(model, optimizer, simulated_io_delay=io_delay) if strategy == "async" else None

    loop_times = []
    stall_times = []

    # Warm-up step to initialize CUDA runtime, graph optimizations, and optimizer state tracks
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    for iteration in range(1, num_iterations + 1):
        # High precision synchronization setup for profiling metrics
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_loop = time.perf_counter()

        # --- Compute Phase (Forward / Backward Pass) ---
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # --- Checkpoint Trigger Phase (Scheduled every 2 iterations) ---
        checkpoint_triggered = (iteration % 2 == 0)
        stall_time = 0.0

        if checkpoint_triggered:
            if torch.cuda.is_available():
                torch.cuda.synchronize() # Finalize ongoing computes before staging state

            checkpoint_start = time.perf_counter()

            if strategy == "baseline":
                # Simulated Synchronous torch.save blocking stall
                _ = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                time.sleep(io_delay)
                stall_time = time.perf_counter() - checkpoint_start

            elif strategy == "async":
                # Trigger Non-blocking state pipeline snapshot
                async_provider.trigger_async_checkpoint(iteration)
                stall_time = time.perf_counter() - checkpoint_start

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end_loop = time.perf_counter()

        loop_duration = end_loop - start_loop
        loop_times.append(loop_duration)
        stall_times.append(stall_time)

        print(f"  Iteration {iteration:02d} | Step Time: {loop_duration:.4f}s | Checkpoint Stall Overhead: {stall_time:.4f}s")

    # Clean up and flush background threads if necessary
    if strategy == "async":
        async_provider.join()
        async_provider.io_queue.put(None) # Poison pill termination signal

    return loop_times, stall_times

In [ ]:
# 4. Benchmarking and Metrics Visualizer
# ==========================================
if __name__ == "__main__":
    NUM_STEPS = 8
    SIMULATED_DISK_DELAY = 0.35  # seconds

    # Execute both strategies under identical network conditions
    base_loop, base_stall = run_simulation(strategy="baseline", num_iterations=NUM_STEPS, io_delay=SIMULATED_DISK_DELAY)
    async_loop, async_stall = run_simulation(strategy="async", num_iterations=NUM_STEPS, io_delay=SIMULATED_DISK_DELAY)

    # Aggregated Summary Calculations
    total_base_time = sum(base_loop)
    total_async_time = sum(async_loop)
    total_base_stall = sum(base_stall)
    total_async_stall = sum(async_stall)
    performance_gain = ((total_base_time - total_async_time) / total_base_time) * 100

    print("\n" + "="*65)
    print(" BENCHMARK PERFORMANCE REPORT (DataStates-LLM POC)")
    print("="*65)
    print(f"Total Iterations Monitored : {NUM_STEPS}")
    print(f"Simulated Disk Write Delay : {SIMULATED_DISK_DELAY:.3f}s")
    print("-"*65)

    # Metrics Comparison Table
    print(f"{'Metric Component':<30} | {'Baseline (Sync)':<15} | {'Proposed (Async)':<15}")
    print(f"{'-'*30} | {'-'*15} | {'-'*15}")
    print(f"{'Total Run Duration':<30} | {total_base_time:>13.4f}s | {total_async_time:>13.4f}s")
    print(f"{'Accumulated Storage Stall':<30} | {total_base_stall:>13.4f}s | {total_async_stall:>13.4f}s")
    print(f"{'Average Loop Step Latency':<30} | {torch.tensor(base_loop).mean().item():>13.4f}s | {torch.tensor(async_loop).mean().item():>13.4f}s")
    print("-"*65)

    # Text-based visual terminal chart
    print("\n Loop Latency Trace Visualizer (Lower is Better)")
    print(f"Baseline (Sync)  : " + "".join(["█" if t > (total_base_time/NUM_STEPS) else "▄" for t in base_loop]))
    print(f"Proposed (Async) : " + "".join(["█" if t > (total_base_time/NUM_STEPS) else "▄" for t in async_loop]))

    print(f"\n Conclusion: The Proposed State Provider reduced training step delays by")
    print(f"   {total_base_stall - total_async_stall:.4f}s total, yielding a {performance_gain:.2f}% training pipeline throughput speedup.\n")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set academic typography and styling rules
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "figure.dpi": 300
})

# ==========================================
# 1. Empirical Data Input (From Your Run)
# ==========================================
iterations = np.arange(1, 9)

# Baseline Strategy Metrics
baseline_step_time = [0.1857, 0.7510, 0.1785, 0.7098, 0.1772, 0.6946, 0.1760, 0.7064]
baseline_stall     = [0.0000, 0.5673, 0.0000, 0.5282, 0.0000, 0.5154, 0.0000, 0.5273]

# Proposed Asynchronous Strategy Metrics
async_step_time    = [0.1877, 0.2181, 0.1781, 0.2185, 0.1758, 0.2141, 0.1778, 0.2124]
async_stall        = [0.0000, 0.0362, 0.0000, 0.0360, 0.0000, 0.0363, 0.0000, 0.0361]

# ==========================================
# Figure 1: Step Latency & Storage Stall Trace
# ==========================================
fig, ax = plt.subplots(figsize=(7, 4))

# Plot Baseline with a distinct color, marker, and line style
ax.plot(iterations, baseline_step_time, color="#D32F2F", linestyle="-", marker="o",
        linewidth=2, markersize=6, label="Baseline (Synchronous) - Total Step")
ax.plot(iterations, baseline_stall, color="#D32F2F", linestyle=":", marker="x",
        linewidth=1.5, markersize=6, label="Baseline (Synchronous) - I/O Stall")

# Plot Proposed Async Strategy
ax.plot(iterations, async_step_time, color="#1976D2", linestyle="-", marker="s",
        linewidth=2, markersize=6, label="Proposed (Asynchronous) - Total Step")
ax.plot(iterations, async_stall, color="#1976D2", linestyle=":", marker="^",
        linewidth=1.5, markersize=6, label="Proposed (Asynchronous) - I/O Stall")

# Formatting Axis Labels and Grid
ax.set_xlabel("Training Iteration Index", fontweight="bold", labelpad=8)
ax.set_ylabel("Execution Time (Seconds)", fontweight="bold", labelpad=8)
ax.set_title("Step Latency Overlap & Storage Stall Trace Comparison", fontweight="bold", pad=12)
ax.set_xticks(iterations)
ax.grid(True)

# Place legend cleanly outside or inside depending on whitespace
ax.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="none")

plt.tight_layout()
plt.savefig("checkpoint_latency_trace.pdf", format="pdf", bbox_inches="tight")
plt.savefig("checkpoint_latency_trace.png", format="png", bbox_inches="tight")
plt.show()


# ==========================================
# Figure 2: Aggregated Macro-Metrics Summary
# ==========================================
categories = ["Total Run\nDuration", "Accumulated\nStorage Stall"]
baseline_totals = [3.5793, 2.1382]
async_totals    = [1.5825, 0.1446]

x = np.arange(len(categories))
width = 0.35  # Bar width configuration

fig, ax = plt.subplots(figsize=(5.5, 4))

# Render grouped bars
rects1 = ax.bar(x - width/2, baseline_totals, width, label="Baseline (Sync)", color="#E57373", edgecolor="#D32F2F")
rects2 = ax.bar(x + width/2, async_totals, width, label="Proposed (Async)", color="#64B5F6", edgecolor="#1976D2")

# Annotate values directly on top of bars for micro-level clarity
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f"{height:.3f}s",
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha="center", va="bottom", fontsize=9)

autolabel(rects1)
autolabel(rects2)

# Macro Chart Formatting
ax.set_ylabel("Cumulative Time (Seconds)", fontweight="bold", labelpad=8)
ax.set_title("Macro Pipeline Performance Metrics Summary", fontweight="bold", pad=12)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylim(0, 4.2)  # Give headroom for labels
ax.grid(axis="y")
ax.legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.savefig("checkpoint_macro_summary.pdf", format="pdf", bbox_inches="tight")
plt.savefig("checkpoint_macro_summary.png", format="png", bbox_inches="tight")
plt.show()

print("📁 Output Vector PDFs ('checkpoint_latency_trace.pdf' & 'checkpoint_macro_summary.pdf') saved successfully.")